# CSV / Excel データ確認・クリーニングノートブック

アップロードした表データを安全な既定値で読み込み、列ごとの型・欠損・重複を確認して、必要な処理だけを選んで別ファイルに書き出します。元ファイルは変更しません。

**使い方**: 上から順に実行し、アップロード画面で CSV または `.xlsx` を1つ選択してください。Colab には `pandas` が標準で入っています。Excel 読み込み時だけ `openpyxl` を必要に応じて導入します。

> 個人情報や機密データを含むファイルを扱う場合は、組織の Colab 利用ルールを確認してください。

In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd

print(f"pandas {pd.__version__}")

## 1. ファイルを1つアップロード

このノートブックはアップロードしたファイルを読み取り、Colab の一時領域で処理します。Drive はマウントしません。

In [ ]:
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError(f"ファイルを1つだけ選択してください（選択数: {len(uploaded)}）")

input_name = next(iter(uploaded))
input_path = Path(input_name)
extension = input_path.suffix.lower()
if extension not in {".csv", ".xlsx"}:
    raise ValueError("対応形式は .csv と .xlsx です")

print(f"読み込み対象: {input_path.name}")

## 2. 読み込み

CSV は UTF-8（BOM 付きも可）を先に試し、日本語 Windows 環境でよく使われる CP932 を次に試します。区切り文字は pandas に推定させます。Excel は先頭シートを読み込みます。

In [ ]:
if extension == ".csv":
    try:
        df = pd.read_csv(input_path, encoding="utf-8-sig", sep=None, engine="python")
    except UnicodeDecodeError:
        df = pd.read_csv(input_path, encoding="cp932", sep=None, engine="python")
else:
    try:
        import openpyxl  # noqa: F401
    except ImportError:
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl"], check=True)
    df = pd.read_excel(input_path, engine="openpyxl")

print(f"行数: {len(df):,} / 列数: {len(df.columns):,}")
display(df.head())

## 3. データの状態を確認

以下の表で列ごとの pandas 型、欠損数・割合、ユニーク値数を確認します。先頭5行と列名も表示します。

In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(1),
    "unique_count": df.nunique(dropna=True),
}).sort_values(["missing_count", "unique_count"], ascending=[False, True])

display(profile)
print(f"完全重複行: {df.duplicated().sum():,}")
print("列名:", list(df.columns))

## 4. 必要な処理を選択

初期設定では、データを削除・補完・変更しません。必要な項目だけ `True` に変更してください。数値の欠損補完を有効にする場合は、中央値または平均値を指定できます。カテゴリ列や文字列の空欄は自動補完しません。

In [ ]:
# 既定値は非破壊です。必要な処理だけ切り替えてください。
DROP_DUPLICATE_ROWS = False
DROP_ALL_EMPTY_COLUMNS = False
TRIM_TEXT_WHITESPACE = False  # 文字列セルの前後空白を削除
NUMERIC_FILL = None           # None / "median" / "mean"

if NUMERIC_FILL not in {None, "median", "mean"}:
    raise ValueError('NUMERIC_FILL は None, "median", "mean" のいずれかです')

clean_df = df.copy()

if TRIM_TEXT_WHITESPACE:
    text_columns = clean_df.select_dtypes(include=["object", "string"]).columns
    for column in text_columns:
        clean_df[column] = clean_df[column].map(
            lambda value: value.strip() if isinstance(value, str) else value
        )

if DROP_DUPLICATE_ROWS:
    clean_df = clean_df.drop_duplicates().copy()

if DROP_ALL_EMPTY_COLUMNS:
    clean_df = clean_df.dropna(axis=1, how="all").copy()

if NUMERIC_FILL is not None:
    numeric_columns = clean_df.select_dtypes(include="number").columns
    for column in numeric_columns:
        fill_value = clean_df[column].median() if NUMERIC_FILL == "median" else clean_df[column].mean()
        if pd.notna(fill_value):
            clean_df[column] = clean_df[column].fillna(fill_value)

print(f"処理前: {df.shape} → 処理後: {clean_df.shape}")
display(clean_df.head())

## 5. 別ファイルとして保存してダウンロード

出力名に時刻を付け、元ファイルを上書きしないようにしています。CSV は UTF-8 BOM 付きで保存するため、日本語版 Excel でも文字化けしにくくなります。

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
output_path = Path(f"{input_path.stem}_cleaned_{timestamp}.csv")
clean_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"保存しました: {output_path.name} ({output_path.stat().st_size:,} bytes)")
files.download(str(output_path))

## 処理の記録

再現性のため、入力ファイル名、処理設定、行列数を記録します。結果を別途共有するときは、入力データに個人情報が含まれていないか確認してください。

In [ ]:
run_summary = {
    "input_file": input_path.name,
    "input_shape": tuple(df.shape),
    "output_shape": tuple(clean_df.shape),
    "drop_duplicate_rows": DROP_DUPLICATE_ROWS,
    "drop_all_empty_columns": DROP_ALL_EMPTY_COLUMNS,
    "trim_text_whitespace": TRIM_TEXT_WHITESPACE,
    "numeric_fill": NUMERIC_FILL,
    "output_file": output_path.name,
}
run_summary